# Complexity-erasing language detection PoC (Llama 3.1 8B, free Colab T4)

Two-step, **article-level** detection of 2 techniques that erase complexity in public debate (a subset of the SemEval / Da San Martino et al. propaganda taxonomy): `black_and_white` (dichotomous reasoning) and `thought_terminating_cliche`.

For each article, two separate model calls:
- **Step 1 (claim extraction)**: read the whole article and list the discrete claims/assertions it makes.
- **Step 2 (instance extraction)**: given the article and the Step 1 claims, pull out every concrete *instance* of the two techniques anywhere in the article — an exact quote, the claim it operates on, and a rationale grounded in the codebook's "distinction from X" guardrails (see `src/codebook.py`). The same technique can be reported multiple times if it occurs more than once.

Runs **Llama-3.1-8B-Instruct** quantized (GGUF, Q4_K_M) locally via `llama-cpp-python`.

**Before running:** In Colab, go to `Runtime > Change runtime type` and select a **T4 GPU**. Free-tier T4 has ~15GB VRAM, which comfortably fits an 8B model at 4-bit quantization.

This uses a public GGUF conversion of Llama 3.1 8B Instruct, so **no Hugging Face account or token is required**.

## 1. Install dependencies
This installs a prebuilt CUDA wheel of `llama-cpp-python` so GPU offload works without a slow from-source compile.

In [ ]:
!pip install -q huggingface_hub
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
# If the prebuilt wheel above fails to install (Colab's CUDA version can drift over time),
# fall back to compiling from source instead (slower, ~5-10 min):
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --no-cache-dir --force-reinstall --upgrade

## 2. Clone this repo (codebook, prompt code, and the labeled/target data live here)
The repo needs to be public (or you need to handle auth yourself) for an anonymous clone to work.

In [ ]:
import os

REPO_URL = "https://github.com/hrauxloh/DAAD_Destructive_polarization"
BRANCH = "claude/concept-language-llama-collab-91hjaq"
REPO_DIR = "/content/DAAD_Destructive_polarization"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
import sys
sys.path.insert(0, REPO_DIR)

## 3. Download the quantized model (GGUF)
Using `bartowski/Meta-Llama-3.1-8B-Instruct-GGUF`, a public community conversion. `Q4_K_M` is a good quality/size tradeoff (~4.9GB) for a T4.

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF"
MODEL_FILE = "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print(model_path)

## 4. Load the model with GPU offload
`n_ctx` is raised to 8192 (vs. the paragraph-level version's 4096) since Step 2's prompt now carries the full article text plus the codebook plus the Step 1 claims. Very long articles are still truncated (see `MAX_ARTICLE_CHARS` in section 8) to stay within budget on a free T4.

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,   # offload all layers to GPU
    n_ctx=8192,
    n_threads=os.cpu_count(),
    verbose=False,
)

## 5. Wire up the codebook, two-step pipeline, and a `generate_fn`
`analyze_article` (in `src/prompting.py`) runs Step 1 then Step 2 and returns `{"claims": [...], "instances": [...]}`.

In [ ]:
from src.prompting import analyze_article, ParseError, VALID_KEYS

def generate_fn(messages, max_tokens=1024, temperature=0.0):
    resp = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return resp["choices"][0]["message"]["content"]

def print_article_result(result):
    print(f'claims ({len(result["claims"])}):')
    for c in result["claims"]:
        print(f'  - {c}')
    if not result["instances"]:
        print("no technique instances found")
    for inst in result["instances"]:
        print(f'  [{inst["technique"]}] "{inst["quote"]}"')
        print(f'    claim: {inst["claim"]}')
        print(f'    -> {inst["rationale"]}')
    print()

## 6. Try it on a single (short) article

In [ ]:
sample_article = (
    "Crime is rising simply because judges have gotten too soft on offenders. "
    "Either we crack down now or our streets will never be safe again. "
    "It is what it is — nothing more to discuss."
)

result = analyze_article(generate_fn, sample_article)
print_article_result(result)

## 7. Evaluate against the real labeled data (validation, lower priority for now)
`oversimplification_data.csv` (repo root) has ~190 SemEval-derived spans labeled `Black-and-White_Fallacy` or `Thought-terminating_Cliches`. Each row's `span_text` is run through the full two-step pipeline as a standalone "article"; there's no source-article context and no negative examples, so this only measures recall/technique-confusion. Each example now costs 2 LLM calls (Step 1 + Step 2), so it's slower than the old paragraph-level eval — start with a small `sample_size`.

In [ ]:
from src.eval import load_labeled_examples, evaluate, print_report

examples = load_labeled_examples(sample_size=15, seed=0)
result = evaluate(generate_fn, examples=examples)
print_report(result)

## 8. Run over unseen news text (Australian climate-change corpus)
`australia_498sample_climatechange.csv` (repo root) has ~480 full news articles with a `full_text` column — the actual "unseen text" target. Each article is run through the two-step pipeline as a whole (no paragraph splitting): Step 1 extracts claims across the full article, Step 2 pulls out technique instances anywhere in it.

`full_text` is truncated to `MAX_ARTICLE_CHARS` characters to keep both calls within the model's context budget on a free T4 — raise it if you have headroom, but very long articles may need it lowered instead.

Two CSVs are saved:
- `aus_claims.csv` — one row per article: `document_id`, `title`, `claims` (Step 1 output, pipe-separated)
- `aus_instances.csv` — one row per detected instance: `document_id`, `title`, `technique`, `quote`, `claim`, `rationale` (Step 2 output) — this is the "coding" table for inspection

In [ ]:
import csv
import pandas as pd

with open("australia_498sample_climatechange.csv", newline="", encoding="utf-8") as f:
    aus_articles = list(csv.DictReader(f))

print(f"loaded {len(aus_articles)} articles")

N_ARTICLES = 3
MAX_ARTICLE_CHARS = 6000   # truncate long articles to fit the context budget

claim_rows = []
instance_rows = []

for article in aus_articles[:N_ARTICLES]:
    text = article["full_text"][:MAX_ARTICLE_CHARS]
    print(f"=== {article['document_id']} : {article['title'][:80]} ===")
    try:
        result = analyze_article(generate_fn, text)
    except ParseError as e:
        print(f"  PARSE_FAIL: {e}")
        claim_rows.append({
            "document_id": article["document_id"], "title": article["title"],
            "claims": None, "parse_error": str(e),
        })
        continue

    claim_rows.append({
        "document_id": article["document_id"],
        "title": article["title"],
        "claims": " | ".join(result["claims"]),
        "parse_error": None,
    })
    for inst in result["instances"]:
        instance_rows.append({
            "document_id": article["document_id"],
            "title": article["title"],
            "technique": inst["technique"],
            "quote": inst["quote"],
            "claim": inst["claim"],
            "rationale": inst["rationale"],
        })

    print_article_result(result)

claims_df = pd.DataFrame(claim_rows)
instances_df = pd.DataFrame(instance_rows)

claims_df.to_csv("aus_claims.csv", index=False)
instances_df.to_csv("aus_instances.csv", index=False)
print(f"saved {len(claims_df)} article rows to aus_claims.csv")
print(f"saved {len(instances_df)} instance rows to aus_instances.csv")
instances_df.head()

In [ ]:
from google.colab import files
files.download("aus_claims.csv")
files.download("aus_instances.csv")

## 9. Run over your own news text
Paste a full article below — it's run through the same two-step pipeline as a single article (not split into paragraphs).

In [ ]:
news_text = """PASTE NEWS TEXT HERE"""

result = analyze_article(generate_fn, news_text)
print_article_result(result)

## Notes / known limitations of this PoC
- Free Colab GPUs are not guaranteed and sessions can disconnect; re-run from the top if that happens.
- Article-level Step 2 asks the model to scan a much longer span of text than the earlier per-paragraph version and report every instance it finds — this is a harder task for an 8B model than a single paragraph YES/NO, so check the `aus_instances.csv` output carefully for both missed and spurious instances before trusting it at scale.
- Currently scoped to 2 techniques by request (`black_and_white`, `thought_terminating_cliche`); `src/codebook.py` previously also covered `causal_oversimplification` and `reductio_ad_hitlerum` and can have them re-added if needed later.
- `MAX_ARTICLE_CHARS` truncation means very long articles are only partially analyzed; instances in the truncated tail won't be found.
- `oversimplification_data.csv` has span-level labels but no source article full-text, so eval in section 7 treats each span as its own standalone "article" and has no negative (`none`) examples — validation is intentionally deprioritized here; section 8 (the real news corpus) is the priority.
- `temperature=0.0` is used for reproducibility.